# 🌊 Pronóstico del nivel del Orinoco a 15 días — Proyecto completo

**Objetivo:** pronosticar el nivel del río (en metros) en **Ciudad Bolívar** y **Palúa** con **15 días de
anticipación**, aprovechando lo que ya está pasando río arriba (Puerto Ayacucho y Caicara).

Este notebook recorre el proyecto de principio a fin y reproduce el análisis del documento
`Nivel_Orinoco_Analisis_y_Modelos.pdf`:

1. Datos y control de calidad (cambios de referencia de la regla).
2. Análisis exploratorio: estacionalidad, propagación aguas abajo y memoria del río.
3. Diseño experimental: pronóstico directo, 79 variables y partición por años.
4. Modelos: líneas base, Ridge y LightGBM paso a paso (los demás se entrenan con `python main.py`).
5. Comparación de resultados y contribución de las variables.

> ⚠️ Mientras no lleguen los datos reales se usan datos **simulados** (`create_dummy_data.py`) con las
> mismas propiedades que describe el PDF.

## La plantilla de 4 pasos

| Paso | Qué se hace |
|------|-------------|
| **PASO 1: Preparación de datos** | Variables conocidas en t, objetivo y(t+h) − y(t), split, escalado |
| **PASO 2: Definición del modelo** | Elegir el modelo y sus hiperparámetros |
| **PASO 3: Entrenamiento** | Ajustar el modelo con train (validación solo para detenerse o elegir) |
| **PASO 4: Predicción e inversión de escala** | Predecir el cambio y reconstruir el nivel: ŷ(t+h) = y(t) + cambio |

## 0. Imports y configuración

Todas las librerías se importan al principio. Si alguna falta: `pip install -r requirements.txt`.
Cambia `ESTACION` para analizar la otra estación objetivo (`"Ciudad_Bolivar"` o `"Palua"`).

In [ ]:
import lightgbm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge

import create_dummy_data
from src.configuracion import (
    CAMBIOS_DE_REFERENCIA, ESTACIONES, FECHA_FIN_TRAIN, FECHA_FIN_VAL, HORIZONTES, NOMBRES_ESTACIONES,
    RUTA_DATOS, SEMILLA, UMBRALES_ALERTA,
)
from src.contribucion import cargar_contribucion
from src.evaluate import calcular_metricas, cargar_metricas, cargar_predicciones
from src.preprocessing import (
    ajustar_climatologias, cambio_a_nivel, corregir_cambios_de_referencia, evaluar_climatologia,
    grupos_de_variables, imputar_huecos, leer_excel, preparar_datos, revisar_calidad,
)

ESTACION = "Ciudad_Bolivar"     # <-- estación que analizamos en este notebook
HORIZONTE = 15                  # días de anticipación

plt.rcParams["figure.figsize"] = (14, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
COLORES = {"Ayacucho": "#7c3aed", "Caicara": "#0284c7", "Ciudad_Bolivar": "#c2410c", "Palua": "#15803d"}

if not RUTA_DATOS.exists():
    create_dummy_data.main()

---
## 1. Datos y control de calidad

### 1.1 Lectura
`leer_excel` convierte `Fecha` en el índice y fuerza **frecuencia diaria**: si una fecha no aparece en el
archivo, se añade como fila vacía (`NaN`).

In [ ]:
crudos = leer_excel()
print("Días:", len(crudos), "| desde", crudos.index.min().date(), "hasta", crudos.index.max().date())
print("\nDías sin dato por estación:")
print(crudos.isna().sum())
crudos.tail()

In [ ]:
fig, ax = plt.subplots()
ax.plot(crudos.index, crudos["Ciudad_Bolivar"], color=COLORES["Ciudad_Bolivar"], linewidth=0.6)
ax.axhline(UMBRALES_ALERTA["Ciudad_Bolivar"], color="red", linestyle="--", label="Umbral de alerta 16,5 m")
ax.set_title("Figura 1. Nivel diario en Ciudad Bolívar (sin corregir)")
ax.set_ylabel("Nivel (m)")
ax.legend()
plt.show()

### 1.2 Cambios de referencia (cero de la regla)

Si se reinstala la regla limnimétrica con otro cero, la serie da un **escalón** que no es del río.
Para detectarlo comparamos cada estación con **dos vecinas** muy correlacionadas:

1. Explicamos la estación con una recta a partir de la vecina; el residuo debería ser suave.
2. Un cambio de referencia es un salto brusco del residuo, **aislado** (los días de alrededor están
   tranquilos) y **sostenido** (el residuo se queda en el nuevo valor).
3. Se **confirma** solo si aparece frente a las dos vecinas; si aparece frente a una sola, el problema
   suele estar en esa vecina (es un "reflejo").

In [ ]:
revision = revisar_calidad(crudos)
revision

Los únicos saltos confirmados son los de Ciudad Bolívar (15-05-1987 y 01-01-2019); los demás son sus reflejos
en Caicara, Palúa y Ayacucho. Tras verificarlos, se declaran en `CAMBIOS_DE_REFERENCIA` y se corrigen
llevando toda la serie a la **referencia actual** (la de los umbrales de alerta vigentes).

In [ ]:
print("Correcciones declaradas:", CAMBIOS_DE_REFERENCIA)
corregidos = corregir_cambios_de_referencia(crudos, CAMBIOS_DE_REFERENCIA)

fig, ax = plt.subplots()
tramo = slice("2018-09-01", "2019-04-30")
ax.plot(crudos.loc[tramo, "Ciudad_Bolivar"], label="Sin corregir", color="gray")
ax.plot(corregidos.loc[tramo, "Ciudad_Bolivar"], label="Corregido", color=COLORES["Ciudad_Bolivar"])
ax.plot(crudos.loc[tramo, "Palua"], label="Palúa (vecina)", color=COLORES["Palua"], linestyle="--")
ax.set_title("Salto del 01-01-2019 en Ciudad Bolívar: antes y después de corregir")
ax.set_ylabel("Nivel (m)")
ax.legend()
plt.show()

print("Saltos confirmados tras corregir:", int(revisar_calidad(corregidos)["Confirmado"].sum()))

### 1.3 Imputación de huecos

Rellenamos los huecos con **interpolación temporal**, pero guardamos qué días tenían dato real
(`observados`). Un día interpolado se rellenó mirando datos posteriores, así que **las métricas solo se
calculan en días observados** (para no evaluar sobre valores inventados).

In [ ]:
niveles, observados = imputar_huecos(corregidos)
print("NaN tras imputar:", int(niveles.isna().sum().sum()))
print("Días observados por estación:")
print(observados.sum())

---
## 2. Análisis exploratorio

### 2.1 Componente estacional

El ciclo anual medio se modela con una **regresión armónica** (senos y cosenos del día del año) ajustada
**solo con train** (1974–2014). Con K = 2 armónicos ya se captura casi toda la forma del hidrograma.

In [ ]:
climatologias = ajustar_climatologias(niveles, observados)
serie = niveles[ESTACION]
climatologia = pd.Series(evaluar_climatologia(climatologias[ESTACION], niveles.index), index=niveles.index)

fig, ax = plt.subplots(figsize=(14, 5))
for anio in range(1974, 2026):
    del_anio = serie[str(anio)]
    ax.plot(del_anio.index.dayofyear, del_anio.to_numpy(), color="lightsteelblue", linewidth=0.5)
anio_2020 = climatologia["2020"]
ax.plot(anio_2020.index.dayofyear, anio_2020.to_numpy(), color="black", linewidth=2.5, label="Climatología armónica")
ax.set_title("Figura 2. Hidrogramas anuales de " + NOMBRES_ESTACIONES[ESTACION] + " (cada línea es un año)")
ax.set_xlabel("Día del año")
ax.set_ylabel("Nivel (m)")
ax.legend()
plt.show()

varianza_explicada = 1 - (serie - climatologia).var() / serie.var()
print("La climatología explica el %.1f %% de la varianza" % (100 * varianza_explicada))

### 2.2 La onda de crecida viaja aguas abajo

Restando a cada estación su media, se ve que el ciclo llega primero a Ayacucho, luego a Caicara y por último
a Ciudad Bolívar y Palúa (casi a la vez). Ese desfase es **información gratis** para pronosticar.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for estacion in ESTACIONES:
    ciclo_medio = niveles[estacion].groupby(niveles.index.dayofyear).mean()
    ax.plot(ciclo_medio.index, ciclo_medio - ciclo_medio.mean(), label=NOMBRES_ESTACIONES[estacion],
            color=COLORES[estacion], linewidth=2)
ax.set_title("Figura 3. Ciclo anual medio (desviación respecto a la media de cada estación)")
ax.set_xlabel("Día del año")
ax.set_ylabel("m")
ax.legend()
plt.show()

maximos = niveles.groupby(niveles.index.year).idxmax()
for estacion in ["Ayacucho", "Caicara"]:
    desfase = (maximos["Ciudad_Bolivar"] - maximos[estacion]).dt.days.median()
    print("Desfase mediano del máximo anual", estacion, "-> Ciudad Bolívar:", desfase, "días")
print("Correlación Ciudad Bolívar - Palúa: %.3f" % niveles["Ciudad_Bolivar"].corr(niveles["Palua"]))

### 2.3 Memoria del río

La **anomalía** (nivel − climatología) cambia despacio: su autocorrelación a 15 días sigue siendo muy alta.
Eso explica por qué incluso la "anomalía amortiguada" (una línea base muy simple) funciona bien.

In [ ]:
anomalia = serie - climatologia
rezagos = range(0, 121)
autocorrelacion = []
for rezago in rezagos:
    autocorrelacion.append(anomalia.autocorr(rezago))

fig, ax = plt.subplots()
ax.plot(list(rezagos), autocorrelacion, color=COLORES[ESTACION])
ax.axvline(15, color="steelblue", label="h = 15")
ax.axhline(1 / np.e, color="gray", linestyle=":", label="1/e")
ax.set_title("Figura 5. Autocorrelación de la anomalía de " + NOMBRES_ESTACIONES[ESTACION])
ax.set_xlabel("Rezago (días)")
ax.legend()
plt.show()
print("Autocorrelación a 15 días: %.2f" % anomalia.autocorr(15))

---
## 3. Diseño experimental

**Pronóstico directo:** ŷ(t + 15) = y(t) + f(x(t)). El modelo aprende el **cambio** que tendrá el río en los
próximos 15 días usando solo información disponible el día t. Predecir el cambio (y no el nivel) ayuda a
los árboles, que no saben extrapolar más allá de lo visto en train.

**79 variables** (todas conocidas en t):

| Grupo | Definición | n |
|-------|------------|---|
| Cíclicas | sen y cos (2πk·d/N), k = 1..3, en t + h; cambio climatológico esperado | 7 |
| Rezagos | y(t − L), L ∈ {0, 1, 2, 3, 5, 7, 10, 15, 20, 30}, para las 4 estaciones | 40 |
| Tendencias | Δw = y(t) − y(t − w), w ∈ {1, 3, 7, 15, 30} | 20 |
| Suavizado | medias móviles de 7 y 30 días y anomalía | 12 |

**Partición por años** (según la fecha objetivo t + h): train 1974–2014, validación 2015–2019, prueba 2020–2025.
La climatología y el escalador Z-Score se ajustan **solo con train**.

In [ ]:
datos = preparar_datos(niveles, observados, ESTACION, HORIZONTE)
grupos = grupos_de_variables(datos["variables"], ESTACION)
print("Variables:", len(datos["variables"]))
for nombre_grupo in grupos:
    print("  ", nombre_grupo, len(grupos[nombre_grupo]))
print("Train:", len(datos["y_train"]), "| Validación:", len(datos["y_val"]), "| Prueba:", len(datos["y_test"]))
print("Fin de train:", FECHA_FIN_TRAIN, "| fin de validación:", FECHA_FIN_VAL)
datos["tabla_test"].head()

---
## 4. Modelos

Guardaremos las predicciones de prueba (en metros) de cada modelo en un diccionario para compararlas.
Las métricas se calculan solo en días observados.

In [ ]:
tabla_test = datos["tabla_test"]
dias_observados = tabla_test["observado"].to_numpy()
nivel_real = tabla_test["nivel_objetivo"].to_numpy()
predicciones = {}
metricas = {}


def registrar(nombre_modelo, nivel_predicho):
    # Guarda la predicción y sus métricas (solo días observados)
    predicciones[nombre_modelo] = nivel_predicho
    metricas[nombre_modelo] = calcular_metricas(nivel_real[dias_observados], nivel_predicho[dias_observados])
    print(nombre_modelo.ljust(20), "MAE = %.1f cm" % (100 * metricas[nombre_modelo]["MAE"]),
          "| NSE = %.3f" % metricas[nombre_modelo]["NSE"])

### 4.1 Líneas base

- **Persistencia:** "dentro de 15 días el río estará igual que hoy" → cambio = 0.
- **Climatología:** "estará en su nivel típico de esa fecha".
- **Anomalía amortiguada:** nivel típico + una fracción φ de la anomalía de hoy (φ se estima con train).

In [ ]:
columna_anomalia = ESTACION + "_anomalia"
tabla_train = datos["tabla_train"]

# Persistencia
registrar("Persistencia", cambio_a_nivel(tabla_test["nivel_actual"], np.zeros(len(tabla_test))))

# Climatología: clim(t+h) - y(t) = cambio climatológico esperado - anomalía de hoy
cambio_climatologia = tabla_test["climatologia_cambio_esperado"] - tabla_test[columna_anomalia]
registrar("Climatologia", cambio_a_nivel(tabla_test["nivel_actual"], cambio_climatologia))

# Anomalía amortiguada: φ = pendiente entre la anomalía de hoy y la de dentro de h días (en train)
anomalia_hoy = tabla_train[columna_anomalia]
anomalia_futura = anomalia_hoy + tabla_train["cambio"] - tabla_train["climatologia_cambio_esperado"]
phi = (anomalia_hoy * anomalia_futura).sum() / (anomalia_hoy ** 2).sum()
print("phi =", round(phi, 3))
cambio_amortiguado = tabla_test["climatologia_cambio_esperado"] - (1 - phi) * tabla_test[columna_anomalia]
registrar("AnomaliaAmortiguada", cambio_a_nivel(tabla_test["nivel_actual"], cambio_amortiguado))

### 4.2 Regresión Ridge

Una suma ponderada de las 79 variables con penalización L2 (alpha), que encoge los pesos para no
sobreajustar con variables muy correlacionadas. Elegimos alpha con **validación**.

In [ ]:
# === PASO 1: PREPARACIÓN DE DATOS ===
# Ya está hecha: x_* son las 79 variables escaladas e y_* el cambio a 15 días.

# === PASO 2: DEFINICIÓN DEL MODELO ===
mejor_alpha = None
mejor_mae = float("inf")
for alpha in [0.1, 1.0, 10.0, 100.0, 1000.0]:
    candidato = Ridge(alpha=alpha).fit(datos["x_train"], datos["y_train"])
    mae_val = calcular_metricas(datos["y_val"], candidato.predict(datos["x_val"]))["MAE"]
    print("alpha =", alpha, "-> MAE validación = %.1f cm" % (100 * mae_val))
    if mae_val < mejor_mae:
        mejor_mae = mae_val
        mejor_alpha = alpha
modelo_ridge = Ridge(alpha=mejor_alpha)

# === PASO 3: ENTRENAMIENTO ===
modelo_ridge.fit(datos["x_train"], datos["y_train"])

# === PASO 4: PREDICCIÓN E INVERSIÓN DE ESCALA ===
cambio_predicho = modelo_ridge.predict(datos["x_test"])
registrar("Ridge", cambio_a_nivel(tabla_test["nivel_actual"], cambio_predicho))

In [ ]:
# ¿Qué variables pesan más? (coeficientes sobre variables escaladas: son comparables entre sí)
pesos = pd.Series(modelo_ridge.coef_, index=datos["variables"])
pesos.abs().sort_values(ascending=False).head(12).sort_values().plot.barh(figsize=(10, 4.5), color="#0284c7")
plt.title("Ridge: las 12 variables con más peso (valor absoluto del coeficiente)")
plt.show()

### 4.3 LightGBM (Boosting)

Construye árboles **uno detrás de otro**, cada uno corrigiendo los errores de los anteriores.
Usamos validación para detenernos cuando deja de mejorar (*early stopping*).

In [ ]:
# === PASO 1: PREPARACIÓN DE DATOS ===
# Mismas 79 variables y mismo objetivo (el cambio a 15 días).

# === PASO 2: DEFINICIÓN DEL MODELO ===
modelo_lgbm = LGBMRegressor(
    n_estimators=3000, learning_rate=0.02, num_leaves=31, min_child_samples=40,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    objective="l1",          # optimiza directamente el MAE
    random_state=SEMILLA, n_jobs=-1, verbose=-1,
)

# === PASO 3: ENTRENAMIENTO ===
parada_temprana = lightgbm.early_stopping(stopping_rounds=150, verbose=False)
modelo_lgbm.fit(datos["x_train"], datos["y_train"], eval_X=datos["x_val"], eval_y=datos["y_val"],
                callbacks=[parada_temprana])
print("Árboles usados:", modelo_lgbm.best_iteration_)

# === PASO 4: PREDICCIÓN E INVERSIÓN DE ESCALA ===
registrar("LightGBM", cambio_a_nivel(tabla_test["nivel_actual"], modelo_lgbm.predict(datos["x_test"])))

### 4.4 El resto de modelos

Estos modelos tardan más y se entrenan con `python main.py` (cada uno en su archivo de `src/models/`):

| Modelo | Idea | Archivo |
|--------|------|---------|
| Random Forest, XGBoost | Más ensamblados de árboles | `tree_models.py` |
| SARIMAX | Anomalía ARMA(p, 0, q) + anomalías aguas arriba de hace 15 días como exógenas | `arima_model.py` |
| Prophet | Ciclo anual + regresores conocidos en t (niveles y tendencias de las 4 estaciones) | `google_model.py` |
| LSTM, CNN 1D, Transformer | Ventanas de 60 días × 10 canales; una sola red predice los 15 horizontes | `neural_models.py` |
| Ensamble | Combina los cambios de todos con pesos ≥ 0 ajustados en validación (2015–2019) | `ensamble.py` |

Si ya ejecutaste `python main.py`, aquí se leen sus resultados.

In [ ]:
tabla_metricas = cargar_metricas()
if tabla_metricas.empty:
    print("Todavía no hay resultados: ejecuta python main.py")
else:
    a_15_dias = tabla_metricas[(tabla_metricas["Estacion"] == ESTACION) & (tabla_metricas["Horizonte"] == HORIZONTE)]
    a_15_dias = a_15_dias.sort_values("MAE").set_index("Modelo")
    a_15_dias["MAE (cm)"] = 100 * a_15_dias["MAE"]
    a_15_dias["RMSE (cm)"] = 100 * a_15_dias["RMSE"]
    columnas = ["MAE (cm)", "RMSE (cm)", "NSE", "Mejora_vs_persistencia"]
    display(a_15_dias[columnas].style.format("{:.2f}").highlight_min(subset=["MAE (cm)"], color="#bbf7d0"))

---
## 5. Comparación de resultados

### 5.1 Nivel real vs. pronóstico a 15 días (modelos entrenados en este notebook)

In [ ]:
COLORES_MODELOS = {"Persistencia": "gray", "Climatologia": "silver", "AnomaliaAmortiguada": "#a16207",
                   "Ridge": "#0284c7", "LightGBM": "#65a30d"}
fechas = pd.DatetimeIndex(tabla_test["fecha_objetivo"])

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(fechas[dias_observados], nivel_real[dias_observados], color="black", linewidth=2.2, label="Observado")
for nombre_modelo in ["Persistencia", "Ridge", "LightGBM"]:
    ax.plot(fechas, predicciones[nombre_modelo], color=COLORES_MODELOS[nombre_modelo], linewidth=1,
            linestyle="--", label=nombre_modelo)
ax.axhline(UMBRALES_ALERTA.get(ESTACION, np.nan), color="red", linestyle=":", label="Umbral de alerta")
ax.set_title("Figura 9. " + NOMBRES_ESTACIONES[ESTACION] + ": nivel observado y pronóstico a 15 días (2020-2025)")
ax.set_ylabel("Nivel (m)")
ax.legend(ncol=5, loc="upper left")
plt.show()

Fíjate en la **persistencia**: es la misma curva observada pero corrida 15 días a la derecha. Los modelos
que usan las estaciones aguas arriba "ven venir" la crecida y la siguen casi sin retraso.

### 5.2 Error según el horizonte (todos los modelos, desde `outputs/`)

In [ ]:
if not tabla_metricas.empty:
    de_la_estacion = tabla_metricas[tabla_metricas["Estacion"] == ESTACION]
    fig, ax = plt.subplots(figsize=(10, 5))
    for nombre_modelo in ["Persistencia", "AnomaliaAmortiguada", "SARIMAX", "Ridge", "LightGBM", "LSTM", "Ensamble"]:
        del_modelo = de_la_estacion[de_la_estacion["Modelo"] == nombre_modelo].sort_values("Horizonte")
        if not del_modelo.empty:
            ax.plot(del_modelo["Horizonte"], 100 * del_modelo["MAE"], marker="o", label=nombre_modelo)
    ax.set_title("Figura 7. MAE en prueba según el horizonte — " + NOMBRES_ESTACIONES[ESTACION])
    ax.set_xlabel("Horizonte (días)")
    ax.set_ylabel("MAE (cm)")
    ax.set_xticks(HORIZONTES)
    ax.legend()
    plt.show()

### 5.3 Contribución de las variables

Se reentrena quitando un grupo de variables cada vez (`python -m src.contribucion`). Cuanto más sube el
error, más información aportaba ese grupo.

In [ ]:
contribucion = cargar_contribucion()
if ESTACION in contribucion:
    tabla_contribucion = 100 * pd.DataFrame(contribucion[ESTACION])
    tabla_contribucion.plot.barh(figsize=(10, 4), color=["#0284c7", "#65a30d"])
    plt.gca().invert_yaxis()
    plt.title("Figura 8. MAE a 15 días (cm) al quitar cada grupo — " + NOMBRES_ESTACIONES[ESTACION])
    plt.xlabel("MAE (cm)")
    plt.show()
    display(tabla_contribucion.round(1))
else:
    print("Ejecuta python -m src.contribucion para ver este análisis")

---
## Conclusiones

- La **persistencia** es una línea base exigente a 1 día, pero a 15 días se equivoca más de un metro: el
  ciclo anual mueve el río muy rápido en la crecida y en la recesión.
- La mayor ganancia viene de **mirar río arriba**: la onda de Ayacucho tarda unos 18 días en llegar a
  Ciudad Bolívar, casi exactamente el horizonte que queremos pronosticar.
- **Predecir el cambio** y no el nivel permite que los árboles funcionen aunque el nivel de partida sea un
  récord.
- Un modelo lineal bien planteado (Ridge) ya captura la mayor parte de la señal; los modelos complejos solo
  se justifican si mejoran claramente esa referencia en los **datos reales**.
- Todas las métricas son sobre datos que ningún modelo vio (2020–2025) y solo en días observados.